In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
PJME_data = pd.read_csv("processed_data/PJME_phase2_preprocessed.csv",parse_dates=["Datetime"])

In [4]:
PJME_data = PJME_data.set_index("Datetime")

# Sort chronologically
PJME_data = PJME_data.sort_index()

print(PJME_data.shape)
print(PJME_data.head())

(145224, 15)
                     PJME_MW  Hour  Day  Week  Month  DayOfWeek  IsWeekend  \
Datetime                                                                     
2002-01-08 01:00:00  29445.0     1    8     2      1          1          0   
2002-01-08 02:00:00  28670.0     2    8     2      1          1          0   
2002-01-08 03:00:00  28375.0     3    8     2      1          1          0   
2002-01-08 04:00:00  28542.0     4    8     2      1          1          0   
2002-01-08 05:00:00  29261.0     5    8     2      1          1          0   

                       Lag_1   Lag_24   Lag_48  Lag_168  Rolling_Mean_24  \
Datetime                                                                   
2002-01-08 01:00:00  31187.0  26862.0  27100.0  30393.0     33452.583333   
2002-01-08 02:00:00  29445.0  25976.0  26097.0  29265.0     33560.208333   
2002-01-08 03:00:00  28670.0  25641.0  25793.0  28357.0     33672.458333   
2002-01-08 04:00:00  28375.0  25666.0  25657.0  27899.0     

In [6]:
# Make sure data is chronological
PJME_data = PJME_data.sort_index()

n = len(PJME_data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = PJME_data.iloc[:train_end].copy()
validation = PJME_data.iloc[train_end:val_end].copy()
test = PJME_data.iloc[val_end:].copy()

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print(train.index.min(), "to", train.index.max())
print(validation.index.min(), "to", validation.index.max())
print(test.index.min(), "to", test.index.max())

Train: (101656, 15)
Validation: (21784, 15)
Test: (21784, 15)
2002-01-08 01:00:00 to 2013-08-13 16:00:00
2013-08-13 17:00:00 to 2016-02-07 08:00:00
2016-02-07 09:00:00 to 2018-08-03 00:00:00


In [5]:
%pip install scikit-learn

^C
Note: you may need to restart the kernel to use updated packages.


In [7]:
# CREATE EVALUATION FUNCTION
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

def evaluate_model(actual, predicted):

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean( np.abs((actual - predicted) / actual)) * 100
    r2 = r2_score(actual, predicted)
    bias = np.mean(predicted - actual)

    return mae, rmse, mape, r2, bias

In [8]:
actual = test["PJME_MW"].values

naive_pred = test["Lag_1"].values

naive_results = evaluate_model(
    actual,
    naive_pred
)

print("Naive Forecast")
print("MAE :", naive_results[0])
print("RMSE:", naive_results[1])
print("MAPE:", naive_results[2])
print("R²  :", naive_results[3])
print("Bias:", naive_results[4])

Naive Forecast
MAE : 1054.5384226955564
RMSE: 1354.4559009673437
MAPE: 3.4266256741954764
R²  : 0.9557950184338169
Bias: -0.20331435916268822


In [9]:
moving_avg_pred = test["Rolling_Mean_24"].values

moving_avg_results = evaluate_model(
    actual,
    moving_avg_pred
)

print("Moving Average")
print("MAE :", moving_avg_results[0])
print("RMSE:", moving_avg_results[1])
print("MAPE:", moving_avg_results[2])
print("R²  :", moving_avg_results[3])
print("Bias:", moving_avg_results[4])

Moving Average
MAE : 3558.373439221447
RMSE: 4468.461840302137
MAPE: 11.71802771572057
R²  : 0.5188753794847885
Bias: -6.741365987270168


In [10]:
previous_day_pred = test["Lag_24"].values

previous_day_results = evaluate_model(
    actual,
    previous_day_pred
)

print("Previous-Day Forecast")
print("MAE :", previous_day_results[0])
print("RMSE:", previous_day_results[1])
print("MAPE:", previous_day_results[2])
print("R²  :", previous_day_results[3])
print("Bias:", previous_day_results[4])

Previous-Day Forecast
MAE : 2206.332124495042
RMSE: 3027.4071711112974
MAPE: 7.013357410061764
R²  : 0.7791572506534058
Bias: -9.834695189129636


In [11]:
previous_week_pred = test["Lag_168"].values

previous_week_results = evaluate_model(
    actual,
    previous_week_pred
)

print("Previous-Week Forecast")
print("MAE :", previous_week_results[0])
print("RMSE:", previous_week_results[1])
print("MAPE:", previous_week_results[2])
print("R²  :", previous_week_results[3])
print("Bias:", previous_week_results[4])

Previous-Week Forecast
MAE : 3434.4158097686377
RMSE: 4735.138843834098
MAPE: 10.698903853644104
R²  : 0.45973490441370857
Bias: -41.495501285347046


In [12]:
baseline_results = pd.DataFrame({
    "Model": [
        "Naive",
        "Moving Average",
        "Previous Day",
        "Previous Week"
    ],

    "MAE": [
        naive_results[0],
        moving_avg_results[0],
        previous_day_results[0],
        previous_week_results[0]
    ],

    "RMSE": [
        naive_results[1],
        moving_avg_results[1],
        previous_day_results[1],
        previous_week_results[1]
    ],

    "MAPE": [
        naive_results[2],
        moving_avg_results[2],
        previous_day_results[2],
        previous_week_results[2]
    ],

    "R2": [
        naive_results[3],
        moving_avg_results[3],
        previous_day_results[3],
        previous_week_results[3]
    ],

    "Bias": [
        naive_results[4],
        moving_avg_results[4],
        previous_day_results[4],
        previous_week_results[4]
    ]
})

print(baseline_results)

            Model          MAE         RMSE       MAPE        R2       Bias
0           Naive  1054.538423  1354.455901   3.426626  0.955795  -0.203314
1  Moving Average  3558.373439  4468.461840  11.718028  0.518875  -6.741366
2    Previous Day  2206.332124  3027.407171   7.013357  0.779157  -9.834695
3   Previous Week  3434.415810  4735.138844  10.698904  0.459735 -41.495501


In [13]:
baseline_results.to_csv(
    "processed_data/phase4_baseline_results.csv",
    index=False
)

print("Baseline results saved.")

Baseline results saved.


### Actual Vs Baseine

In [14]:
plt.figure(figsize=(16, 6))

plt.plot(
    test.index[:500],
    actual[:500],
    label="Actual",
    color="black"
)

plt.plot(
    test.index[:500],
    naive_pred[:500],
    label="Naive",
    alpha=0.8
)

plt.plot(
    test.index[:500],
    previous_day_pred[:500],
    label="Previous Day",
    alpha=0.8
)

plt.plot(
    test.index[:500],
    previous_week_pred[:500],
    label="Previous Week",
    alpha=0.8
)

plt.title("Actual vs Baseline Forecasts")
plt.xlabel("Date")
plt.ylabel("Electricity Demand (MW)")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig(
    "images/phase4_baseline_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

NameError: name 'plt' is not defined